# 1. Prompts & Output Parsers

Three of LangChain's most foundational building blocks:
- `ChatPromptTemplate` turns user input into a well-structured message for the model.
- Invoking a chat model directly.
- Two output-parser styles: a plain `StrOutputParser` (just the raw text) and a
  `PydanticOutputParser` (the model is instructed, via format instructions embedded
  in the prompt, to emit JSON that gets parsed into a validated Pydantic object).

**Prerequisites:** Ollama running locally with `llama3.2` pulled
(`ollama pull llama3.2`).

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

Project root on sys.path: /Users/arun.sarma/my_exp/ai_projects/ai_tutorial


## Part 1 — `ChatPromptTemplate` + `StrOutputParser`

`StrOutputParser` is the difference between getting the model's raw `AIMessage`
(content + metadata like token usage and stop reason) and getting just the text.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from models.chat_models.ollama_models import SupportedModel, get_chat_model

llm = get_chat_model(SupportedModel.llama3_2)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful teaching assistant. Answer in a {style} way."),
        ("human", "Explain: {topic}"),
    ]
)

# Without a parser: the raw AIMessage object.
chain_raw = prompt | llm
raw_result = chain_raw.invoke({"style": "concise", "topic": "black holes"})
print(type(raw_result))
print(raw_result)

<class 'langchain_core.messages.ai.AIMessage'>
content="**What is a Black Hole?**\n\nA black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime.\n\n**Key Characteristics:**\n\n1. **Event Horizon**: The point of no return around a black hole. Once you cross the event horizon, you're trapped.\n2. **Singularity**: The center of a black hole, where the density and gravity are infinite.\n3. **Gravitational Pull**: Black holes are characterized by their incredibly strong gravity, which warps spacetime around them.\n\n**Types of Black Holes:**\n\n1. **Stellar Black Holes**: Formed from the collapse of individual stars.\n2. **Supermassive Black Holes**: Found at the centers of galaxies, with masses millions or even billions of times that of the sun.\n3. **Intermediate-Mass Black Holes**: Black holes with masse

In [3]:
# With StrOutputParser: just the text.
chain = prompt | llm | StrOutputParser()
text_result = chain.invoke({"style": "concise", "topic": "black holes"})
print(type(text_result))
print(text_result)

<class 'langchain_core.messages.base.TextAccessor'>
**What is a Black Hole?**

A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime.

**Key Characteristics:**

1. **Event Horizon**: The point of no return around a black hole. Once you cross the event horizon, you're trapped.
2. **Singularity**: The center of a black hole, where the density and gravity are infinite.
3. **Gravitational Pull**: Black holes are characterized by their incredibly strong gravity, which warps spacetime around them.

**Types of Black Holes:**

1. **Stellar Black Holes**: Formed from the collapse of individual stars.
2. **Supermassive Black Holes**: Found at the centers of galaxies, with masses millions or even billions of times that of the sun.
3. **Intermediate-Mass Black Holes**: Black holes with masses that fall betwee

## Under the Hood — the `Runnable` Interface

Every object you just used — `prompt`, `llm`, `StrOutputParser()` — implements
the same shared interface: `Runnable`. That's what makes `|` (the pipe
operator) work at all: `a | b` isn't special-cased for prompts or models
specifically, it just builds a `RunnableSequence` that calls `a.invoke(x)` and
feeds the result into `b.invoke(...)`. Anything implementing `Runnable`
(`.invoke()`, `.batch()`, `.ainvoke()`, `.stream()`) can be composed with `|`
— including things you write yourself (`RunnableLambda`, previewed below).

**The class hierarchy for what we're actually using here** (verified against
the installed library via `__mro__`, not just recalled):

| Object | Class | Inherits from | Takes | Returns |
|---|---|---|---|---|
| `prompt` | `ChatPromptTemplate` | `BaseChatPromptTemplate` → `BasePromptTemplate` → `RunnableSerializable` → `Runnable` | a `dict` of template variables | a `ChatPromptValue` (a `Runnable` itself) |
| `llm` | `ChatOllama` | `BaseChatModel` → `BaseLanguageModel` → `RunnableSerializable` → `Runnable` | a `PromptValue`, string, or message list | an `AIMessage` |
| `StrOutputParser()` | `StrOutputParser` | `BaseTransformOutputParser` → `BaseOutputParser` → `RunnableSerializable` → `Runnable` | an `AIMessage` (or string) | a plain `str` |
| `PydanticOutputParser(...)` | `PydanticOutputParser` | `JsonOutputParser` → `BaseCumulativeTransformOutputParser` → `BaseOutputParser` → `Runnable` | a `str` of (hopefully) JSON | a validated Pydantic instance |

Every chat model you'll see anywhere in this project (`ChatOllama` is the only
one used here, since everything runs against local Ollama models) is a
`BaseChatModel` subclass — *that's* the actual contract LangChain cares about,
not "which provider." Swap `ChatOllama` for `ChatOpenAI`/`ChatAnthropic`/etc.
and every chain in this notebook keeps working unchanged, because they all
implement the same `Runnable` interface with the same input/output shapes.

In [4]:
from langchain_core.runnables import Runnable

print("prompt is a Runnable:", isinstance(prompt, Runnable), f"({type(prompt).__name__})")
print("llm is a Runnable:   ", isinstance(llm, Runnable), f"({type(llm).__name__})")
print("chain is a Runnable: ", isinstance(chain, Runnable), f"({type(chain).__name__})")
print()

# `|` built a RunnableSequence — you can inspect its actual steps directly:
print("chain.steps: ", [type(s).__name__ for s in chain.steps])
print("chain.first: ", type(chain.first).__name__)
print("chain.middle:", [type(s).__name__ for s in chain.middle])
print("chain.last:  ", type(chain.last).__name__)

prompt is a Runnable: True (ChatPromptTemplate)
llm is a Runnable:    True (ChatOllama)
chain is a Runnable:  True (RunnableSequence)

chain.steps:  ['ChatPromptTemplate', 'ChatOllama', 'StrOutputParser']
chain.first:  ChatPromptTemplate
chain.middle: ['ChatOllama']
chain.last:   StrOutputParser


**Other well-known `Runnable` variants** — not used in *this* notebook, but
you'll build with them in notebook 2 (`02-lcel-chains.ipynb`):

- **`RunnableParallel`** — runs multiple Runnables against the *same* input
  concurrently, collecting results into a dict (`{"a": chain_a, "b": chain_b}`
  runs both branches, not one after the other).
- **`RunnableLambda`** — lifts a plain Python function into a `Runnable`, so
  you can drop custom logic into a `|` pipeline.
- **`RunnablePassthrough`** — a no-op `Runnable` that just returns its input
  unchanged; useful inside a `RunnableParallel` when you want the *original*
  input to survive alongside derived branches.

All three, like everything above, are just classes implementing `.invoke()` —
there's no special magic beyond that one shared interface.

## Part 2 — `PydanticOutputParser`

Instead of trusting the model to format its own answer, we embed format
instructions in the prompt and parse the raw text into a validated Pydantic object.
`format="json"` on the model call helps a smaller model like `llama3.2` emit
syntactically valid JSON more reliably.

In [5]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class ConceptExplanation(BaseModel):
    summary: str = Field(description="A one or two sentence summary of the topic.")
    key_points: list[str] = Field(description="3 to 5 short bullet points about the topic.")


parser = PydanticOutputParser(pydantic_object=ConceptExplanation)

structured_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You explain topics for a JSON API. Respond with ONLY a single JSON "
            'object containing REAL VALUES for the fields "summary" and '
            '"key_points" — never return the schema/field definitions '
            "themselves. Example of a correctly filled response: "
            '{{"summary": "Example summary sentence.", '
            '"key_points": ["point one", "point two", "point three"]}}'
            "\n\n{format_instructions}",
        ),
        ("human", "Explain: {topic}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

structured_chain = structured_prompt | llm.bind(format="json") | parser
explanation = structured_chain.invoke({"topic": "retrieval-augmented generation"})
print(explanation)
print()
print("summary:", explanation.summary)
print("key_points:", explanation.key_points)

summary='Retrieval-Augmented Generation (RAG) is a technique used in natural language processing (NLP) to improve the performance of language models.' key_points=['Improves language model performance by retrieving relevant information from a knowledge base', "Enhances generation capabilities by incorporating retrieved information into the model's output", 'Can be used for a variety of NLP tasks, including question answering, text summarization, and more']

summary: Retrieval-Augmented Generation (RAG) is a technique used in natural language processing (NLP) to improve the performance of language models.
key_points: ['Improves language model performance by retrieving relevant information from a knowledge base', "Enhances generation capabilities by incorporating retrieved information into the model's output", 'Can be used for a variety of NLP tasks, including question answering, text summarization, and more']


## 🧪 Playground

A few starter experiments — modify the cells above or add new ones below.

**1. Change the `style`** to `"like I'm five"` or `"as a rhyming poem"` and re-run Part 1.

In [ ]:
# TODO: re-run chain.invoke with a different style


**2. Add a field** to `ConceptExplanation` (e.g. `difficulty: str`) and see if the model fills it in correctly.

In [ ]:
# TODO: define ConceptExplanationV2 with an extra field and re-run the structured chain


**3. Try a topic prone to long answers** (e.g. `"the history of the internet"`) — does `PydanticOutputParser` still succeed, or does it fail to parse?

In [ ]:
# TODO: try a broader topic and see whether parsing still succeeds
chain1=prompt|llm1|StrOutputParser()
chain2=prompt|llm2|StrOutputParser()
chain3=prompt|llm3|StrOutputParser()
chain4=prompt|llm4|StrOutputParser()
RunnableSequence(prompt,llm1,StrOutputParser())
parallelChain = RunnableParallel(
    chain1,chain2,chain3,chain4,
)
##graph.add_edge(START, node1)
##graph.add_edge(START, node2)
##graph.add_edge(START, node3)
parallelChain.invoke({"",""})

Runnable RunnableSerialiable RUnnableSequence